# Import Statements

In [ ]:
import custom_cmap
import os
import sys
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from moria import reduce
from astroquery.ogle import Ogle
from astropy.coordinates import SkyCoord
from astropy import units as u
from pathlib import Path
from astropy.visualization import PercentileInterval
from astropy.coordinates import SkyCoord
from astropy import units as u
from re import A

import inspect
import numpy as np
#from new_flystar.flystar import match
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import matplotlib.font_manager
import matplotlib.ticker
from matplotlib.ticker import FormatStrFormatter
import pandas as pd

#import smplotlib



from astropy.io import fits
from astropy.visualization import LogStretch, ImageNormalize
import plotly.express as px
import numpy as np
from astropy.visualization import ImageNormalize, AsinhStretch, SqrtStretch, LogStretch, PowerStretch

plt.rcParams.update({'font.size':25})
plt.rc('text', usetex=True)
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2


# Understanding the main directory. 


The directory structure for your data analysis should be organized as follows. 

<li>00.DATA</li>
<li>01.XYM</li>
<li>02.CMD</li>
<li>03.LOC_TRANS</li>
<li>04.PSF_EXTRACT</li>
<li>05.COORD_TRANS(OPTIONAL)</li>
<li>06.FIT</li>
<li>07.CALIBRATION</li>

This directory structure has been set up in "MORIA/data". 

All necessary scripts are included within the corresponding folders under MORIA/data. Therefore, the simplest way to run MORIA on your target is to copy all eight folders from MORIA/data to the location where you intend to perform your analysis. Begin by placing your exposures in data/00.DATA.

Note: chmod +x program.src is a useful command to run whenever script execution fails due to permission issues

# Notebook Goals

This notebook involves manual steps to calibrate HST magnitudes with OGLE-III magnitudes

In [ ]:
#This is the directory where you are processing your data. This does not point at MORIA.
directory = os.getcwd()

# Step 0 

We assume you ran the output_stacks.ipynbm, cmd_diagram.ipynb, creating_psf.ipynb, fitting_psfs.ipynb (in that order) notebooks correctly

# Step 1: Enter coordinates of your target

The goal here is to calibrate the HST photometry to the OGLE-III database. To obtain location of your event you will first need to obtain the OGLE-III field number, chip number, and pixel coordinates at the OGLE web page using the “OGLE Field Finder” under the “Sky Coverage”. MORIA automates this with the scripts below.

Once MORIA does that, it chooses the appropriate OGLE-III catalog file at http://www.astrouw.edu.pl/ogle/ogle3/maps/blg/maps/. It will download the appropriate photometry map for your event from here. 

For example, consider OGLE-2012-BLG-0563: the OGLE-III catalog file is blg226.7.map, and the OGLE reference image corresponding to this photometry catalog file is blg226.I.7.fts, which can be obtained from http://www.astrouw.edu.pl/ogle/ogle3/maps/blg/ref_images/ .

The routines in this module compare the HST photometry to the OGLE photometry in order to enable the HST photometry to be calibrated to OGLE. Special efforts are required to avoid blending effects, where several HST stars will be merged together to make an object that is considered to be a single star in the lower angular resolution OGLE images. Also, PSF-fitting magnitudes are dependent on the details of the PSF model. 

In [ ]:
ogle_ra = "17:51:31.86"
ogle_dec = "-29:33:55.0"

ogle_coord = SkyCoord(ogle_ra, ogle_dec, unit = (u.deg, u.deg))
ogle_ra_deg = np.float64(ogle_coord.ra)
ogle_dec_deg = np.float64(ogle_coord.dec)

ogle_band = "I"


# Step 2: Download the OGLE map and reference image 

In [ ]:
ogle_field_number, ogle_chip_number = reduce.get_chip_number(ogle_ra_deg, ogle_dec_deg)

reduce.download_ogle_map_and_reference(
    directory=directory,
    ogle_field_number=ogle_field_number,
    ogle_chip_number=ogle_chip_number,
    ogle_band=ogle_band,
    destination_subdir="07.CALIBRATION",
    overwrite=False,
)

# Step 3: Designing calibration input

When calibrating the module below, we require running the program psf_star_mags_mcmc.xOg  using the script run_psf_star_Imags_mcmc.src and the input file IN_psf_star_mags_mcmc_I. We will first design the input for this script.

The inputs in this file are:
1. The PSF file name (already entered)
2. The input file name tag (already entered)
3. The output file name tag (already entered)
4. The number of Markov Chain steps (e.g. 100,000)
5. The maximum size of MCMC coordinate steps in pixels and the error bar fudge factor (e.g. 0.02)
6. The maximum distance in x and y from the star center for pixels to be included in the fit, and the χ2 threshold to define an outlier pixel. (e.g. 16)
7. The sky model to use: 0 for annulus, 1 for fit sky on the same pixels as the PSF peak. The annulus sky model is needed for determining the PSF model, but it is more sensitive to blending than the fit sky model. (e.g. 1.0)
8. A list of star numbers to produce PIX_SHOW files for, ending with a 0 to exit the program. These PIX_SHOW are the same as the ones produced in module 06, and they are useful to tell you how well each star was fit. But, they are large files that are time consuming to produce, and it is unlikely that you’ll want them for every star. (e.g. 10)


In [ ]:
# Default priors you can use in the step below:
mcmc_defaults = {
    "markov_chain_steps": 100000,
    "maximum_size_mcmc": 0.02,
    "fudge": 1.0,
    "maximum_distance_x": 2.5,
    "maximum_distance_y": 2.5,
    "chi2cut": 16,
    "sky_model": 1,
    "star_numbers_pix_show": 10,
}

In [ ]:
reduce.calibration_input_file_one(directory)

# Step 4: Recreate calibration input for V magnitude

Repeat this procedure for the V band using script run_psf_star_Vmags_mcmc.src, which uses the input file IN_psf_star_mags_mcmc_V.

In [ ]:
reduce.calibration_input_file_two(directory)

# Step 5: Create new calibration map

When you finish running this step, you should see two new files in 07.CALIBRATION:
1. MATCHUP.F814W_cal.XYM
2. MATCHUP.F814W_cal_only.XYM

In [ ]:
reduce.calibration_new_matchup(directory)

# Step 6

This is a manual step. Notice that the 07.CALIBRATION Setup has four IN.* files in it already. The first is IN.cal_star_num_2_MATCHUP which is used to re-create matchup files used for calibration. Then we have IN.fit_HST_Iogle_col_1 and IN.fit_HST_Iogle_col_2 used in Step 6 below. We also have  IN.VI_HST_ogle_man_match4_backup.


As of now, you need to manually edit 'IN.VI_HST_ogle_man_match4'. This step instructs you on how to edit that file.

An important part of this input file is a list of stars that have been matched in the OGLE reference image and the HST image, outputq_F814W.fits. At this point, navigate to the 07.CALIBRATION folder using your preferred terminal and run 'python starlist2reg.py'. You will be prompted to enter the name of your starlist (e.g. 'blg194.1.map').

Then, open the *.fits file for your map (e.g. 'blg194.1.fits') in DS9. 'starlist2reg.py' created a '.reg' file that you can load to DS9. These will overlay the positions from the OGLE map on your OGLE images. While loading into DS9, load them in the 'xy' format with the 'image' option. Look at the image below.

![ds9](ds9.png)

 Choose ones that look best by eye and edit 'IN.VI_HST_ogle_man_match4.'

In 'IN.VI_HST_ogle_man_match4', edit the first row to your matching radius in arcseconds; enter the coordinates (format: hh mm ss, without colons) for your event in the second row; change your map number in the seventh row. Then, you need to enter information about the matching calibrated stars. The first two entries are the HST x and y coordinates (found in the MATCHUP file), then the OGLE x and y coordinates (found in the starlist), followed by the HST I magnitudes, and then the OGLE I magnitudes for those stars. 

In the OGLE map, the V magnitude is written before the I magnitude. To make it easier to get OGLE x, y, V and I magnitudes, we recommend opening a file called: 'New_{blg_map_name}.map' (e.g. New_blg194.1.map). 'New_{blg_map_name}.map' is a version of the OGLE map that only reports the OGLE x, y, V and I magnitude.s 

Loading the '.reg' file into DS9 makes this process quicker.

However, this is the only part of the pipeline that has been left manual. Once the 'IN.VI_HST_ogle_man_match4' file has been created, proceed with the notebook. 

In [ ]:
reduce.calibration_hst_ogle_match(directory)

# Step 7

The final calibration program is used to select constraints to put on the parameters in the VI_HST_ogle_Cal_matches4.dat file to select the calibration stars and calibrate the HST photometry. This is done program fit_HST_IV_ogle_col.xOg.

This program calculates both I band and V band offsets between the calibrated OGLE photometry and the HST photometry, as well as 2-color fits to calibrate both the HST I and V bands to the OGLE I and V band. 

The results of these calibration attempts have been routed to the run_fit_HST_IV_ogle_col_1.log and run_fit_HST_IV_ogle_col_2.log files.

If this step fails, you should open ``VI_HST_ogle_Cal_matches4.dat" and remove the spacing between the columns with headers Vo-Vhfs and lg_c2Vmx

In [ ]:
reduce.fit_calibration(directory)

You can now open the log_files present in 07.CALIBRATION/log_files to note down the OGLE-III calibrated HST magnitudes. The log file of concern will be labelled 'run_fit_VI_HST_ogle_man_match4.log'.

# If you'd like to celebrate run the cell below

In [ ]:
reduce.notebook_complete("Calibration notebook")

# If you'd like to go further and derive the source/lens calibrated magnitudes

In [ ]:
def flux_to_mag_error(flux, flux_error):
    return 2.5 / np.log(10) * flux_error / flux

def cal_mag_error(I_inst_err, V_inst_err, color_term):
    """Propagate instrumental mag errors through m_cal = zp + (1-k)*I_inst + k*V_inst."""
    return np.sqrt((1 - color_term)**2 * I_inst_err**2 + color_term**2 * V_inst_err**2)

What you have found in this notebook are the I- and V- band zeropoints for calibration + the color terms associated with the magnitudes. If you'd like to plot the "n" number of stars you found in 06.FIT on a CMD diagram around this target, follow these steps.

1. Find the flux1, flux2, flux1_error and flux2_error values for the F814W and F606W filters from 06.FIT for your PSF fitting. For example, if the 2star-fit gave you the best PSF fit, go to 06.FIT/F814W/2star-fit/log_files/run_mcmc_expand_average.log and enter flux1_814, flux2_814, flux1_814_error, flux_814_error, A0_814_error.

In [ ]:
flux1_814 = ...
flux2_814 = ...
flux3_814 = ... #If 3star-fit is your best-fit, flux3_814 = A0 - flux1_814 - flux2_814.
flux1_814_error = ...
flux2_814_error = ...
A0_814_error = ...
flux3_814_error = np.sqrt(flux1_814_error**2 + flux2_814_error**2 + A0_814_error**2)

2. Repeat for F606W filter. For example, if the 2star-fit gave you the best PSF fit, go to 06.FIT/F606W/2star-fit/log_files/run_mcmc_expand_average.log and enter flux1_606, flux2_606, flux1_606_error, flux_606_error, A0_606_error.

In [ ]:
flux1_606 = ...
flux2_606 =  ...
flux3_606 = ... #If 3star-fit is your best-fit, flux3_814 = A0 - flux1_814 - flux2_814.
flux1_606_error = ...
flux2_606_error = ...
A0_606_error = ...
flux3_606_error = np.sqrt(flux1_606_error**2 + flux2_606_error**2 + A0_606_error**2)

In [ ]:
star1_Imag_inst = -2.5*np.log10(flux1_814)
star2_Imag_inst = -2.5*np.log10(flux2_814)
#star3_Imag_inst = -2.5*np.log10(flux3_814)

star1_Imag_inst_error = flux_to_mag_error(flux1_814, flux1_814_error)
star2_Imag_inst_error = flux_to_mag_error(flux2_814, flux2_814_error)
star3_Imag_inst_error = flux_to_mag_error(flux3_814, flux3_814_error)

star1_Vmag_inst = -2.5*np.log10(flux1_606)
star2_Vmag_inst = -2.5*np.log10(flux2_606)
#star3_Vmag_inst = -2.5*np.log10(flux3_606)

star1_Vmag_inst_error = flux_to_mag_error(flux1_606, flux1_606_error)
star2_Vmag_inst_error = flux_to_mag_error(flux2_606, flux2_606_error)
#star3_Vmag_inst_error = flux_to_mag_error(flux3_606, flux3_606_error)

3. Now open fit_HST_IV_ogle_col.log in 07.CALIBRATIONS, scroll to the bottom and enter the I_zeropoint (I_0), V_Zeropoint (V_0), I_color_term(last column in row with I_0) and V_color_term(last column in row with V_0)

In [ ]:
I_zeropoint = ...
I_color_term = ...
V_zeropoint = ...
V_color_term = ...

In [ ]:
# Star 1 calibrated magnitudes
star1_I_mag = I_zeropoint + star1_Imag_inst + I_color_term*(star1_Vmag_inst - star1_Imag_inst)
star1_V_mag = V_zeropoint + star1_Imag_inst + V_color_term*(star1_Vmag_inst - star1_Imag_inst)
star1_I_mag_error = cal_mag_error(star1_Imag_inst_error, star1_Vmag_inst_error, I_color_term)
star1_V_mag_error = cal_mag_error(star1_Imag_inst_error, star1_Vmag_inst_error, V_color_term)

# Star 2 calibrated magnitudes
star2_I_mag = I_zeropoint + star2_Imag_inst + I_color_term*(star2_Vmag_inst - star2_Imag_inst)
star2_V_mag = V_zeropoint + star2_Imag_inst + V_color_term*(star2_Vmag_inst - star2_Imag_inst)
star2_I_mag_error = cal_mag_error(star2_Imag_inst_error, star2_Vmag_inst_error, I_color_term)
star2_V_mag_error = cal_mag_error(star2_Imag_inst_error, star2_Vmag_inst_error, V_color_term)


# Star 3 calibrated magnitudes. Uncomment if required
#star3_I_mag = I_zeropoint + star3_Imag_inst + I_color_term*(star3_Vmag_inst - star3_Imag_inst)
#star3_V_mag = V_zeropoint + star3_Imag_inst + V_color_term*(star3_Vmag_inst - star3_Imag_inst)
#star3_I_mag_error = cal_mag_error(star3_Imag_inst_error, star3_Vmag_inst_error, I_color_term)
#star3_V_mag_error = cal_mag_error(star3_Imag_inst_error, star3_Vmag_inst_error, V_color_term)

In [ ]:
print(f'Star 1 calibrated I magnitudes is: I = {star1_I_mag:.4f} +/- {star1_I_mag_error:.4f}')
print(f'Star 1 calibrated V magnitudes is: V = {star1_V_mag:.4f} +/- {star1_V_mag_error:.4f}')

In [ ]:
print(f'Star 2 calibrated I magnitudes is: I = {star2_I_mag:.4f} +/- {star2_I_mag_error:.4f}')
print(f'Star 2 calibrated V magnitudes is: V = {star2_V_mag:.4f} +/- {star2_V_mag_error:.4f}')

In [ ]:
#print(f'Star 3 calibrated I magnitudes is: I = {star3_I_mag:.4f} +/- {star3_I_mag_error:.4f}')
#print(f'Star 3 calibrated V magnitudes is: V = {star3_V_mag:.4f} +/- {star3_V_mag_error:.4f}')

In [ ]:
# Instrumental magnitude errors (from flux uncertainties)
print('Instrumental magnitude errors:')
print(f'  Star 1: I = +/- {star1_Imag_inst_error:.4f}, V = +/- {star1_Vmag_inst_error:.4f}')
print(f'  Star 2: I = +/- {star2_Imag_inst_error:.4f}, V = +/- {star2_Vmag_inst_error:.4f}')
#print(f'  Star 3: I = +/- {star3_Imag_inst_error:.4f}, V = +/- {star3_Vmag_inst_error:.4f}')